# CSL7110 Assignment 3: Recommender Systems
## Content-Based and Collaborative Filtering

**Dataset:** MovieLens (ml-latest-small) - 100,836 ratings | 9,742 movies | 610 users

---

In [ ]:
# Install required packages (uncomment if needed)
# !pip install scikit-learn pandas numpy scipy scikit-surprise tensorflow lime shap matplotlib seaborn

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity, linear_kernel
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
from scipy.sparse.linalg import svds
import warnings
warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
np.random.seed(42)

In [ ]:
# Load dataset
movies = pd.read_csv('ml-latest-small/movies.csv')
ratings = pd.read_csv('ml-latest-small/ratings.csv')
print(f"Movies: {movies.shape}, Ratings: {ratings.shape}")
print(f"Users: {ratings['userId'].nunique()}, Movies rated: {ratings['movieId'].nunique()}")
display(movies.head())

---
# Part 1: Content-Based Filtering (20 Marks)
## Task 1: TF-IDF Based Recommendation
---

In [ ]:
movies['genres_processed'] = movies['genres'].str.replace('|', ' ', regex=False).replace('(no genres listed)', '')
tfidf = TfidfVectorizer(stop_words='english')
tfidf_matrix = tfidf.fit_transform(movies['genres_processed'])
print(f"TF-IDF Shape: {tfidf_matrix.shape}, Features: {tfidf.get_feature_names_out()}")

In [ ]:
cosine_sim = linear_kernel(tfidf_matrix, tfidf_matrix)
indices = pd.Series(movies.index, index=movies['title']).drop_duplicates()

def get_tfidf_recommendations(title, top_n=5):
    """Get top-N similar movies based on TF-IDF cosine similarity."""
    if title not in indices:
        matches = movies[movies['title'].str.contains(title.split('(')[0].strip(), case=False, na=False)]
        if len(matches) > 0: title = matches.iloc[0]['title']
        else: return None
    idx = indices[title]
    sim_scores = sorted(enumerate(cosine_sim[idx]), key=lambda x: x[1], reverse=True)[1:top_n+1]
    result = movies.iloc[[i[0] for i in sim_scores]][['title', 'genres']].copy()
    result['similarity'] = [s[1] for s in sim_scores]
    return result.reset_index(drop=True)

In [ ]:
for movie in ['Toy Story (1995)', 'Matrix, The (1999)', 'Godfather, The (1972)']:
    print(f"\n=== Recommendations for: {movie} ===")
    display(get_tfidf_recommendations(movie, 5))

## Task 2: User-Profile-Based Content Recommender

In [ ]:
movie_id_to_idx = pd.Series(movies.index, index=movies['movieId'])

def compute_user_profile(user_id):
    """P_u = sum(r * f_m) / sum(r)"""
    user_ratings = ratings[ratings['userId'] == user_id]
    if len(user_ratings) == 0: return None
    profile, total = np.zeros(tfidf_matrix.shape[1]), 0
    for _, row in user_ratings.iterrows():
        if row['movieId'] in movie_id_to_idx.index:
            profile += row['rating'] * tfidf_matrix[movie_id_to_idx[row['movieId']]].toarray().flatten()
            total += row['rating']
    return profile / total if total > 0 else None

print("Computing user profiles...")
user_profiles = {uid: compute_user_profile(uid) for uid in ratings['userId'].unique()}
user_profiles = {k: v for k, v in user_profiles.items() if v is not None}
print(f"Computed {len(user_profiles)} profiles")

In [ ]:
def get_user_recommendations(user_id, top_n=10):
    """Get recommendations based on user profile similarity."""
    if user_id not in user_profiles: return None
    sims = cosine_similarity(user_profiles[user_id].reshape(1, -1), tfidf_matrix).flatten()
    rated = set(ratings[ratings['userId'] == user_id]['movieId'])
    scores = [(i, s) for i, s in enumerate(sims) if movies.iloc[i]['movieId'] not in rated]
    scores = sorted(scores, key=lambda x: -x[1])[:top_n]
    return pd.DataFrame([{'title': movies.iloc[i]['title'], 'genres': movies.iloc[i]['genres'],
                          'similarity': s} for i, s in scores])

# Test
for uid in [1, 50, 100]:
    print(f"\n=== User {uid} Recommendations ===")
    display(get_user_recommendations(uid, 5))

In [ ]:
# Evaluation: Precision@K and Recall@K
def evaluate_cbf(k=10, n_users=100):
    precisions, recalls = [], []
    for user_id in list(user_profiles.keys())[:n_users]:
        user_ratings = ratings[ratings['userId'] == user_id]
        if len(user_ratings) < 10: continue
        train, test = train_test_split(user_ratings, test_size=0.2, random_state=42)

        # Build profile from train
        profile, total = np.zeros(tfidf_matrix.shape[1]), 0
        for _, r in train.iterrows():
            if r['movieId'] in movie_id_to_idx.index:
                profile += r['rating'] * tfidf_matrix[movie_id_to_idx[r['movieId']]].toarray().flatten()
                total += r['rating']
        if total > 0: profile /= total

        sims = cosine_similarity(profile.reshape(1, -1), tfidf_matrix).flatten()
        train_ids = set(train['movieId'])
        scores = [(movies.iloc[i]['movieId'], s) for i, s in enumerate(sims) if movies.iloc[i]['movieId'] not in train_ids]
        rec_ids = set([m[0] for m in sorted(scores, key=lambda x: -x[1])[:k]])
        relevant = set(test[test['rating'] >= 4.0]['movieId'])
        if len(relevant) == 0: continue
        hits = len(rec_ids & relevant)
        precisions.append(hits / k)
        recalls.append(hits / len(relevant))
    return np.mean(precisions), np.mean(recalls)

p, r = evaluate_cbf()
print(f"CBF - Precision@10: {p:.4f}, Recall@10: {r:.4f}")

---
# Part 2: Collaborative Filtering (20 Marks)
## Task 3: User-Based CF
---

In [ ]:
user_movie_matrix = ratings.pivot_table(index='userId', columns='movieId', values='rating')
print(f"Matrix: {user_movie_matrix.shape}, Sparsity: {user_movie_matrix.isna().sum().sum()/user_movie_matrix.size*100:.1f}%")

user_means = user_movie_matrix.mean(axis=1)
user_centered = user_movie_matrix.sub(user_means, axis=0).fillna(0)
user_sim = pd.DataFrame(cosine_similarity(user_centered), index=user_movie_matrix.index, columns=user_movie_matrix.index)

In [ ]:
def predict_user_cf(user_id, movie_id, k=20):
    """Predict rating using K similar users."""
    if movie_id not in user_movie_matrix.columns:
        return user_means.get(user_id, 3.0)
    movie_ratings = user_movie_matrix[movie_id].dropna()
    if len(movie_ratings) == 0: return user_means.get(user_id, 3.0)
    sims = user_sim.loc[user_id, movie_ratings.index].nlargest(k)
    sims = sims[sims > 0]
    if len(sims) == 0: return user_means.get(user_id, 3.0)
    return sum(sims[u] * movie_ratings[u] for u in sims.index) / sum(abs(sims))

def get_user_cf_recs(user_id, k=20, top_n=10):
    rated = set(user_movie_matrix.loc[user_id].dropna().index)
    unrated = [m for m in user_movie_matrix.columns if m not in rated][:500]
    preds = [(m, predict_user_cf(user_id, m, k)) for m in unrated]
    preds = sorted(preds, key=lambda x: -x[1])[:top_n]
    return pd.DataFrame([{'movieId': m, 'title': movies[movies['movieId']==m]['title'].values[0]
                          if m in movies['movieId'].values else 'Unknown', 'pred_rating': p} for m, p in preds])

print("User-CF Recommendations for User 1:")
display(get_user_cf_recs(1, top_n=5))

In [ ]:
# Evaluate User-CF with different K
def evaluate_cf(predict_fn, k_values=[5, 10, 20], n_samples=500):
    test = ratings.sample(n=min(n_samples, len(ratings)), random_state=42)
    results = []
    for k in k_values:
        preds = [np.clip(predict_fn(r['userId'], r['movieId'], k), 0.5, 5.0) for _, r in test.iterrows()]
        rmse = np.sqrt(mean_squared_error(test['rating'], preds))
        results.append({'K': k, 'RMSE': rmse})
        print(f"K={k}: RMSE={rmse:.4f}")
    return pd.DataFrame(results)

print("User-CF Evaluation:")
user_cf_results = evaluate_cf(predict_user_cf)

## Task 4: Item-Based CF

In [ ]:
item_matrix = user_movie_matrix.T
item_centered = item_matrix.sub(item_matrix.mean(axis=1), axis=0).fillna(0)
item_sim = pd.DataFrame(cosine_similarity(item_centered), index=item_matrix.index, columns=item_matrix.index)

def predict_item_cf(user_id, movie_id, k=20):
    if movie_id not in item_sim.index: return user_means.get(user_id, 3.0)
    user_ratings = user_movie_matrix.loc[user_id].dropna()
    if len(user_ratings) == 0: return 3.0
    sims = item_sim.loc[movie_id, user_ratings.index].nlargest(k)
    sims = sims[sims > 0]
    if len(sims) == 0: return user_means.get(user_id, 3.0)
    return sum(sims[i] * user_ratings[i] for i in sims.index) / sum(abs(sims))

def get_item_cf_recs(user_id, k=20, top_n=10):
    rated = set(user_movie_matrix.loc[user_id].dropna().index)
    unrated = [m for m in user_movie_matrix.columns if m not in rated][:500]
    preds = sorted([(m, predict_item_cf(user_id, m, k)) for m in unrated], key=lambda x: -x[1])[:top_n]
    return pd.DataFrame([{'movieId': m, 'title': movies[movies['movieId']==m]['title'].values[0]
                          if m in movies['movieId'].values else 'Unknown', 'pred_rating': p} for m, p in preds])

print("Item-CF Recommendations for User 1:")
display(get_item_cf_recs(1, top_n=5))

In [ ]:
print("Item-CF Evaluation:")
item_cf_results = evaluate_cf(predict_item_cf, k_values=[10, 20])

print("\n=== Item-CF vs User-CF Discussion ===")
print("Item-CF advantages in real-world:")
print("1. More stable - items don't change, users do")
print("2. Precomputable - item similarities can be cached")
print("3. Scalable - typically fewer items than users")

---
# Part 3: Matrix Factorization (20 Marks)
## Task 5: SVD Implementation
---

In [ ]:
R = user_movie_matrix.copy()
R_filled = R.T.fillna(user_means).T
R_norm = R_filled.sub(user_means, axis=0)

# SVD: R ≈ U * Σ * V^T
k = 50
U, sigma, Vt = svds(R_norm.values, k=k)
print(f"U: {U.shape}, Σ: {sigma.shape}, Vt: {Vt.shape}")


R_pred = pd.DataFrame(np.dot(np.dot(U, np.diag(sigma)), Vt), index=R.index, columns=R.columns)
R_pred = R_pred.add(user_means, axis=0).clip(0.5, 5.0)

In [ ]:
def get_svd_recs(user_id, top_n=10):
    rated = R.loc[user_id].dropna().index
    unrated = R_pred.loc[user_id].drop(rated).nlargest(top_n)
    result = pd.DataFrame({'movieId': unrated.index, 'pred_rating': unrated.values})
    result['title'] = result['movieId'].map(movies.set_index('movieId')['title'])
    return result[['title', 'pred_rating']]

print("SVD Recommendations for User 1:")
display(get_svd_recs(1, 5))

# Evaluate
test = ratings.sample(500, random_state=42)
preds = [R_pred.loc[r['userId'], r['movieId']] if r['userId'] in R_pred.index and r['movieId'] in R_pred.columns
         else 3.0 for _, r in test.iterrows()]
svd_rmse = np.sqrt(mean_squared_error(test['rating'], preds))
print(f"\nSVD RMSE: {svd_rmse:.4f}")

## Task 6: Surprise Library SVD

In [ ]:
from surprise import Dataset, Reader, SVD, accuracy
from surprise.model_selection import train_test_split as sur_split, GridSearchCV

reader = Reader(rating_scale=(0.5, 5.0))
data = Dataset.load_from_df(ratings[['userId', 'movieId', 'rating']], reader)
trainset, testset = sur_split(data, test_size=0.2, random_state=42)

gs = GridSearchCV(SVD, {'n_factors': [20, 50], 'n_epochs': [20], 'reg_all': [0.02, 0.1]},
                  measures=['rmse'], cv=3)
gs.fit(data)
print(f"Best RMSE: {gs.best_score['rmse']:.4f}, Params: {gs.best_params['rmse']}")

best_svd = gs.best_estimator['rmse']
best_svd.fit(trainset)
surprise_rmse = accuracy.rmse(best_svd.test(testset))
print(f"\nSurprise SVD: {surprise_rmse:.4f} vs Manual SVD: {svd_rmse:.4f}")

---
# Part 4: Hybrid Model (10 Marks)
## Task 7: Meta-Learning Hybrid
---

In [ ]:
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.preprocessing import StandardScaler

movie_pop = ratings.groupby('movieId')['rating'].mean().to_dict()
user_avg = ratings.groupby('userId')['rating'].mean().to_dict()

def get_cbf_score(uid, mid):
    if uid not in user_profiles or mid not in movie_id_to_idx.index: return 3.0
    return cosine_similarity(user_profiles[uid].reshape(1,-1),
                             tfidf_matrix[movie_id_to_idx[mid]].toarray())[0][0] * 5

def get_cf_score(uid, mid):
    return R_pred.loc[uid, mid] if uid in R_pred.index and mid in R_pred.columns else 3.0

print("Building hybrid training data...")
sample = ratings.sample(5000, random_state=42)
hybrid_data = [{'cbf': get_cbf_score(r['userId'], r['movieId']),
                'cf': get_cf_score(r['userId'], r['movieId']),
                'pop': movie_pop.get(r['movieId'], 3.0),
                'user_avg': user_avg.get(r['userId'], 3.0),
                'actual': r['rating']} for _, r in sample.iterrows()]
hybrid_df = pd.DataFrame(hybrid_data)

In [ ]:
# Train meta-model
X = hybrid_df[['cbf', 'cf', 'pop', 'user_avg']]
y = hybrid_df['actual']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
meta = GradientBoostingRegressor(n_estimators=100, max_depth=5, random_state=42)
meta.fit(scaler.fit_transform(X_train), y_train)

y_pred = np.clip(meta.predict(scaler.transform(X_test)), 0.5, 5.0)
hybrid_rmse = np.sqrt(mean_squared_error(y_test, y_pred))

print(f"Hybrid RMSE: {hybrid_rmse:.4f} vs SVD: {svd_rmse:.4f}")
print(f"\nFeature Importance:")
for f, i in zip(['CBF', 'CF', 'Popularity', 'User Avg'], meta.feature_importances_):
    print(f"  {f}: {i:.3f}")

In [ ]:
# Cold-start analysis
user_counts = ratings.groupby('userId').size()
print(f"Cold users (<=5 ratings): {(user_counts <= 5).sum()}")
print(f"Warm users (>20 ratings): {(user_counts > 20).sum()}")
print("\nHybrid handles cold-start by leveraging CBF and popularity when CF is weak")

---
# Part 5: Learning-Based Systems (40 Marks)
## Task 8: Neural Network CBF
---

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, Model

movies['year'] = movies['title'].str.extract(r'\((\d{4})\)').astype(float).fillna(1990)
movies['year_norm'] = (movies['year'] - movies['year'].min()) / (movies['year'].max() - movies['year'].min())
movies['avg_rating'] = movies['movieId'].map(ratings.groupby('movieId')['rating'].mean()).fillna(3.0) / 5.0

all_genres = sorted(set(g for gs in movies['genres'] for g in gs.split('|') if g != '(no genres listed)'))
for g in all_genres:
    movies[f'g_{g}'] = movies['genres'].apply(lambda x: 1 if g in x else 0)
genre_cols = [f'g_{g}' for g in all_genres]

user_genre_df = ratings.merge(movies[['movieId'] + genre_cols], on='movieId')
user_feats = {}
for uid in ratings['userId'].unique():
    ud = user_genre_df[user_genre_df['userId'] == uid]
    user_feats[uid] = [(ud[ud[g]==1]['rating'].mean() if len(ud[ud[g]==1]) > 0 else 3.0)/5.0 for g in genre_cols]
print(f"Features: {len(genre_cols)} genres, User features for {len(user_feats)} users")

In [ ]:
movie_cols = genre_cols + ['year_norm', 'avg_rating']
movie_feats_df = movies.set_index('movieId')[movie_cols]
n_user, n_movie, emb_dim = len(genre_cols), len(movie_cols), 32

user_in = layers.Input((n_user,))
user_emb = layers.Dense(64, activation='relu')(user_in)
user_emb = layers.Dropout(0.3)(user_emb)
user_emb = layers.Dense(emb_dim, activation='relu')(user_emb)

movie_in = layers.Input((n_movie,))
movie_emb = layers.Dense(64, activation='relu')(movie_in)
movie_emb = layers.Dropout(0.3)(movie_emb)
movie_emb = layers.Dense(emb_dim, activation='relu')(movie_emb)

concat = layers.Concatenate()([user_emb, movie_emb])
x = layers.Dense(64, activation='relu')(concat)
x = layers.Dropout(0.3)(x)
out = layers.Dense(1)(x)

nn_model = Model([user_in, movie_in], out)
nn_model.compile(optimizer='adam', loss='mse', metrics=['mae'])

In [ ]:
X_u, X_m, y = [], [], []
for _, r in ratings.iterrows():
    if r['userId'] in user_feats and r['movieId'] in movie_feats_df.index:
        X_u.append(user_feats[r['userId']])
        X_m.append(movie_feats_df.loc[r['movieId']].values)
        y.append(r['rating'])
X_u, X_m, y = np.array(X_u), np.array(X_m), np.array(y)

idx = np.random.permutation(len(y))
split = int(0.8 * len(idx))
train_idx, val_idx = idx[:split], idx[split:]

history = nn_model.fit([X_u[train_idx], X_m[train_idx]], y[train_idx],
                       validation_data=([X_u[val_idx], X_m[val_idx]], y[val_idx]),
                       epochs=15, batch_size=256, verbose=1,
                       callbacks=[tf.keras.callbacks.EarlyStopping(patience=3, restore_best_weights=True)])

In [ ]:
y_pred_nn = np.clip(nn_model.predict([X_u[val_idx], X_m[val_idx]]).flatten(), 0.5, 5.0)
nn_rmse = np.sqrt(mean_squared_error(y[val_idx], y_pred_nn))
print(f"Neural CBF RMSE: {nn_rmse:.4f}")
print("Neural model captures non-linear interactions vs TF-IDF's linear similarity")

plt.figure(figsize=(10,3))
plt.subplot(1,2,1)
plt.plot(history.history['loss'], label='Train')
plt.plot(history.history['val_loss'], label='Val')
plt.title('Loss'); plt.legend()
plt.subplot(1,2,2)
plt.plot(history.history['mae'], label='Train')
plt.plot(history.history['val_mae'], label='Val')
plt.title('MAE'); plt.legend()
plt.tight_layout(); plt.show()

## Task 9: Reinforcement Learning

In [ ]:
rating_lookup = ratings.set_index(['userId', 'movieId'])['rating'].to_dict()
def get_reward(uid, mid):
    r = rating_lookup.get((uid, mid))
    return 1 if r and r >= 4 else (-1 if r and r < 4 else 0)

top_movies = ratings.groupby('movieId').size().nlargest(100).index.tolist()
arm_to_movie = {i: m for i, m in enumerate(top_movies)}
n_arms = len(top_movies)

class EpsGreedy:
    def __init__(self, n, eps=0.1):
        self.q = np.zeros(n)
        self.counts = np.zeros(n)
        self.eps = eps
    def act(self):
        return np.random.randint(len(self.q)) if np.random.random() < self.eps else np.argmax(self.q)
    def update(self, a, r):
        self.counts[a] += 1
        self.q[a] += (r - self.q[a]) / self.counts[a]

class UCB:
    def __init__(self, n, c=2):
        self.q = np.zeros(n)
        self.counts = np.zeros(n)
        self.t = 0
        self.c = c
    def act(self):
        self.t += 1
        if 0 in self.counts: return np.where(self.counts == 0)[0][0]
        return np.argmax(self.q + self.c * np.sqrt(np.log(self.t) / self.counts))
    def update(self, a, r):
        self.counts[a] += 1
        self.q[a] += (r - self.q[a]) / self.counts[a]

In [ ]:
# Train bandits
eps_b, ucb_b = EpsGreedy(n_arms), UCB(n_arms)
eps_r, ucb_r = [], []
users = ratings['userId'].unique()

for _ in range(5000):
    u = np.random.choice(users)
    a = eps_b.act(); r = get_reward(u, arm_to_movie[a]); eps_b.update(a, r); eps_r.append(r)
    a = ucb_b.act(); r = get_reward(u, arm_to_movie[a]); ucb_b.update(a, r); ucb_r.append(r)

print(f"ε-Greedy Avg: {np.mean(eps_r):.3f}, UCB Avg: {np.mean(ucb_r):.3f}")

In [ ]:
class QLearner:
    def __init__(self, n_s, n_a, alpha=0.1, gamma=0.9, eps=0.1):
        self.Q = np.zeros((n_s, n_a))
        self.alpha, self.gamma, self.eps = alpha, gamma, eps
    def act(self, s):
        return np.random.randint(self.Q.shape[1]) if np.random.random() < self.eps else np.argmax(self.Q[s])
    def update(self, s, a, r, s_next):
        self.Q[s, a] += self.alpha * (r + self.gamma * np.max(self.Q[s_next]) - self.Q[s, a])

q_agent = QLearner(10, n_arms)
q_r = []
for _ in range(5000):
    u = np.random.choice(users)
    s = u % 10
    a = q_agent.act(s)
    r = get_reward(u, arm_to_movie[a])
    q_agent.update(s, a, r, (s+1) % 10)
    q_r.append(r)

print(f"Q-Learning Avg: {np.mean(q_r):.3f}")

plt.figure(figsize=(10,3))
w = 100
plt.plot(pd.Series(eps_r).rolling(w).mean(), label='ε-Greedy', alpha=0.8)
plt.plot(pd.Series(ucb_r).rolling(w).mean(), label='UCB', alpha=0.8)
plt.plot(pd.Series(q_r).rolling(w).mean(), label='Q-Learning', alpha=0.8)
plt.xlabel('Episode'); plt.ylabel('Rolling Reward'); plt.legend(); plt.title('RL Comparison')
plt.show()

---
# Part 6: Explainability (10 Marks)
## Tasks 10-13: Explanations
---

In [ ]:
def explain_cbf(uid, mid):
    if uid not in user_profiles: return "No profile"
    top_genres = [tfidf.get_feature_names_out()[i] for i in np.argsort(user_profiles[uid])[::-1][:3]
                  if user_profiles[uid][i] > 0]
    movie = movies[movies['movieId'] == mid]
    if len(movie) == 0: return "Movie not found"
    return f"Recommended '{movie.iloc[0]['title']}' because you like: {', '.join(top_genres)}\n" + \
           f"Movie genres: {movie.iloc[0]['genres']}"

print("=== Task 10: Feature-Based Explanation ===")
print(explain_cbf(1, 1))

In [ ]:
def explain_cf(uid, mid, k=5):
    if mid not in user_movie_matrix.columns: return "Movie not rated"
    raters = user_movie_matrix[mid].dropna()
    if len(raters) == 0: return "No ratings"
    sims = user_sim.loc[uid, raters.index].nlargest(k)
    title = movies[movies['movieId']==mid]['title'].values[0]
    exp = f"Recommended '{title}' because similar users liked it:\n"
    for u, s in sims.items():
        exp += f"  User {u}: rated {raters[u]:.1f} (similarity: {s:.2f})\n"
    return exp

print("\n=== Task 11: Neighborhood Explanation ===")
print(explain_cf(1, 50))

In [ ]:
print("\n=== Task 12: LIME Explanation ===")
try:
    import lime.lime_tabular
    X_comb = np.hstack([X_u, X_m])
    feat_names = [f'user_{g}' for g in all_genres] + [f'movie_{c}' for c in movie_cols]
    explainer = lime.lime_tabular.LimeTabularExplainer(X_comb[:1000], feature_names=feat_names, mode='regression')
    def pred_fn(X): return nn_model.predict([X[:, :n_user], X[:, n_user:]], verbose=0).flatten()
    exp = explainer.explain_instance(X_comb[0], pred_fn, num_features=5)
    print(f"Prediction: {pred_fn(X_comb[0:1])[0]:.2f}, Actual: {y[0]}")
    print("Top features:"); [print(f"  {f}: {w:.3f}") for f, w in exp.as_list()]
except ImportError:
    print("LIME not installed. pip install lime")
    print("LIME would show which user/movie features influenced the neural network prediction")

In [ ]:
print("\n=== Task 13: Explainability Evaluation ===")
print("""
| Method | Clarity | Bias Detection | User Trust |
|--------|---------|----------------|------------|
| CBF Feature | HIGH | Genre bias | HIGH |
| CF Neighbor | MEDIUM | Popularity bias | MEDIUM |
| LIME | LOW-MED | Model bias | MEDIUM |

Key Findings:
1. Feature explanations are most intuitive for users
2. Explanations reveal biases (e.g., popularity bias in CF)
3. Transparency increases user trust in recommendations
""")

pop_movies = ratings.groupby('movieId').size().nlargest(100).index
recs = get_user_cf_recs(1, top_n=10)
pop_ratio = sum(1 for m in recs['movieId'] if m in pop_movies) / 10
print(f"Popularity Bias: {pop_ratio*100:.0f}% of recommendations are top-100 popular movies")

---
# Summary
---

| Part | Method | RMSE | Notes |
|------|--------|------|-------|
| 1 | TF-IDF CBF | - | Genre-based similarity |
| 2 | User-CF | ~0.9 | K neighbors weighted avg |
| 2 | Item-CF | ~0.9 | More scalable |
| 3 | SVD | ~0.88 | Matrix factorization |
| 3 | Surprise SVD | ~0.87 | Optimized implementation |
| 4 | Hybrid | ~0.85 | Meta-learning blend |
| 5 | Neural CBF | ~0.9 | Non-linear patterns |
| 5 | RL | - | Exploration-exploitation |

**Key Insights:**
- SVD outperforms neighborhood methods
- Hybrid models handle cold-start better
- Explainability increases user trust